
# CIFAR-10 Image Classification using ANN & CNN

## Objective
Build and compare:
1. Artificial Neural Network (ANN)
2. Convolutional Neural Network (CNN)

Evaluate:
- Accuracy
- Loss Curves
- Confusion Matrix
- Classification Report
- Training Strategies (Data Augmentation, Dropout, Batch Normalization)

Designed for **Google Colab**.


In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow Version:", tf.__version__)


In [ ]:

# Load Dataset

(X_train, y_train), (X_test, y_test) = cifar10.load_data()

class_names = [
    'airplane','automobile','bird','cat','deer',
    'dog','frog','horse','ship','truck'
]

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)


In [ ]:

# Visualize Sample Images

plt.figure(figsize=(10,5))

for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(X_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:

# Normalize Data

X_train = X_train / 255.0
X_test = X_test / 255.0


## ANN Model

In [ ]:

ann_model = models.Sequential([
    layers.Flatten(input_shape=(32,32,3)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()


In [ ]:

ann_history = ann_model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)


In [ ]:

ann_loss, ann_acc = ann_model.evaluate(X_test, y_test, verbose=0)

print("ANN Test Accuracy:", ann_acc)


In [ ]:

# ANN Learning Curves

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(ann_history.history['accuracy'])
plt.plot(ann_history.history['val_accuracy'])
plt.title("ANN Accuracy")

plt.subplot(1,2,2)
plt.plot(ann_history.history['loss'])
plt.plot(ann_history.history['val_loss'])
plt.title("ANN Loss")

plt.show()


## CNN Model

In [ ]:

cnn_model = models.Sequential([

    layers.Conv2D(32,(3,3),activation='relu',padding='same',
                  input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Conv2D(64,(3,3),activation='relu',padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Conv2D(128,(3,3),activation='relu',padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(256,activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10,activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()


In [ ]:

# Data Augmentation

datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

datagen.fit(X_train)


In [ ]:
cnn_history = cnn_model.fit(
    datagen.flow(X_train, y_train, batch_size=64),
    epochs=5,
    validation_data=(X_test, y_test)
)


Epoch 1/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 332s 416ms/step - accuracy: 0.3609 - loss: 1.7585 - val_accuracy: 0.4642 - val_loss: 1.5885
Epoch 2/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 331s 424ms/step - accuracy: 0.4885 - loss: 1.4262 - val_accuracy: 0.3564 - val_loss: 2.1798
Epoch 3/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 320s 410ms/step - accuracy: 0.5620 - loss: 1.2403 - val_accuracy: 0.5966 - val_loss: 1.1980
Epoch 4/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 327s 418ms/step - accuracy: 0.6073 - loss: 1.1162 - val_accuracy: 0.6224 - val_loss: 1.1276
Epoch 5/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 326s 417ms/step - accuracy: 0.6392 - loss: 1.0325 - val_accuracy: 0.6782 - val_loss: 0.9528
Epoch 6/20
144/782 ━━━━━━━━━━━━━━━━━━━━ 4:08 390ms/step - accuracy: 0.6557 - loss: 1.0040

In [ ]:

cnn_loss, cnn_acc = cnn_model.evaluate(X_test, y_test, verbose=0)

print("CNN Test Accuracy:", cnn_acc)


In [ ]:

# CNN Learning Curves

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(cnn_history.history['accuracy'])
plt.plot(cnn_history.history['val_accuracy'])
plt.title("CNN Accuracy")

plt.subplot(1,2,2)
plt.plot(cnn_history.history['loss'])
plt.plot(cnn_history.history['val_loss'])
plt.title("CNN Loss")

plt.show()


In [ ]:

# Predictions

pred = cnn_model.predict(X_test)
pred_classes = np.argmax(pred, axis=1)

cm = confusion_matrix(y_test, pred_classes)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.title("CNN Confusion Matrix")
plt.show()

print(classification_report(y_test, pred_classes,
                            target_names=class_names))


In [ ]:

# Performance Comparison

comparison = pd.DataFrame({
    "Model":["ANN","CNN"],
    "Test Accuracy":[ann_acc, cnn_acc]
})

comparison



# Analysis

### ANN
- Uses flattened pixels only.
- Ignores spatial image features.
- Faster but lower accuracy.

### CNN
- Learns edges, textures, and object structures.
- Data augmentation improves generalization.
- Batch normalization stabilizes training.
- Usually achieves much higher accuracy than ANN.

### Expected Results
| Model | Typical Accuracy |
|---------|---------|
| ANN | 45–60% |
| CNN | 75–85% |

### Conclusion
CNN significantly outperforms ANN on CIFAR-10 because convolutional layers extract meaningful image features.
